In [1]:
import torch
from torch.utils.data import TensorDataset,random_split
import matplotlib.pyplot as plt
X=torch.randn(1000,5)

y=(2*X[:,0]+3*X[:,1]-X[:,2]+4*X[:,3]+0.5*X[:,4]).unsqueeze(1)

dataset=TensorDataset(X,y)

In [2]:
train_size=int(0.8*len(dataset))

val_size=len(dataset)-train_size

train_dataset, val_dataset=random_split(dataset,[train_size,val_size],generator=torch.Generator().manual_seed(42))

print("Train:",len(train_dataset))
print("Validation:",len(val_dataset))

Train: 800
Validation: 200


In [3]:
from torch.utils.data import DataLoader

train_daloader=DataLoader(train_dataset,batch_size=32,shuffle=True)
val_daloader=DataLoader(val_dataset,batch_size=32,shuffle=True)

print("Train batches:",len(train_daloader))
print("Validation batches:",len(val_daloader))

Train batches: 25
Validation batches: 7


In [4]:
from torch import nn
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer=nn.Sequential(
            nn.Linear(5,32),
            nn.ReLU(),
            nn.Linear(32,16),
            nn.ReLU(),
            nn.Linear(16,1)
        )

    def forward(self,X):
        return self.layer(X)

In [5]:
model=Model()
print(model)

Model(
  (layer): Sequential(
    (0): Linear(in_features=5, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [6]:
loss_fn=nn.MSELoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [7]:
indices=torch.randperm(len(X))

train_size=int(0.8*len(X))

train_indices=indices[:train_size]
val_indices=indices[train_size:]

X_train=X[train_indices]
y_train=y[train_indices]

X_val=X[val_indices]
y_val=y[val_indices]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

X_train: torch.Size([800, 5])
y_train: torch.Size([800, 1])
X_val: torch.Size([200, 5])
y_val: torch.Size([200, 1])


In [8]:
epochs = 100

train_losses = []
val_losses = []

for epoch in range(epochs):

    # ==========================
    # TRAINING
    # ==========================

    model.train()

    train_pred = model(X_train)

    train_loss = loss_fn(train_pred, y_train)

    optimizer.zero_grad()

    train_loss.backward()

    optimizer.step()

    train_losses.append(train_loss.item())


    # ==========================
    # VALIDATION
    # ==========================

    model.eval()

    with torch.inference_mode():

        val_pred = model(X_val)

        val_loss = loss_fn(val_pred, y_val)

    val_losses.append(val_loss.item())


    # ==========================
    # LOGGING
    # ==========================

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {train_loss.item():.6f} | "
            f"Val Loss: {val_loss.item():.6f}"
        )

Epoch [10/100] | Train Loss: 29.826242 | Val Loss: 29.534950
Epoch [20/100] | Train Loss: 29.115498 | Val Loss: 28.842234
Epoch [30/100] | Train Loss: 28.281008 | Val Loss: 28.021694
Epoch [40/100] | Train Loss: 27.194935 | Val Loss: 26.923384
Epoch [50/100] | Train Loss: 25.724926 | Val Loss: 25.442417
Epoch [60/100] | Train Loss: 23.845404 | Val Loss: 23.557396
Epoch [70/100] | Train Loss: 21.566483 | Val Loss: 21.260557
Epoch [80/100] | Train Loss: 18.895149 | Val Loss: 18.578625
Epoch [90/100] | Train Loss: 15.901943 | Val Loss: 15.588082
Epoch [100/100] | Train Loss: 12.719165 | Val Loss: 12.413894
